# 1. Structured Output

Models can be requested to produce structured output in variety of ways.

## Pydantic

In [14]:
from langchain_ollama import ChatOllama

model = ChatOllama(model="llama3.1:8b", temperature=0)

In [15]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The release year of the movie")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The rating of the movie on a scale of 1 to 10")

model_with_structured_output = model.with_structured_output(Movie)


In [16]:
response = model_with_structured_output.invoke(
    "Provide information about the movie 'Inception'.")

In [17]:
response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

In [18]:
### Nested structures

class Actor(BaseModel):
    name: str = Field(..., description="The name of the actor")
    role: str = Field(..., description="The role played by the actor")

class Movie(BaseModel):
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The release year of the movie")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The rating of the movie on a scale of 1 to 10")
    actors: list[Actor] = Field(..., description="A list of actors in the movie")
    genres: list[str] = Field(..., description="A list of genres the movie belongs to") 
    budget: float | None = Field(..., description="The budget of the movie in USD")

model_with_nested_structured_output = model.with_structured_output(Movie)

In [21]:
response = model_with_nested_structured_output.invoke("Give me details about the movie 'Inception'")

In [22]:
response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8, actors=[Actor(name='Leonardo DiCaprio', role='Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Ellen Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito')], genres=['Action', 'Adventure', 'Science Fiction', 'Thriller'], budget=160000000.0)

## TypedDict

In [23]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with its details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]


model_withtypedict=model.with_structured_output(MovieDict)
response=model_withtypedict.invoke("Please provide the details of the movie avengers")
response

{'title': 'The Avengers',
 'year': 2012,
 'director': 'Joss Whedon',
 'rating': 8.1}

In [24]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Avengers")
response

{'title': 'The Avengers',
 'year': 2012,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Bruce Banner / Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Clint Barton / Hawkeye'}],
 'genres': ['Action, Adventure, Sci-Fi'],
 'budget': 220000000.0}

## Dataclass

In [31]:
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class Contact(BaseModel):
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person

agent = create_agent(model, response_format=Contact)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})
result["structured_response"]

Contact(name='John Doe', email='john@example.com', phone='(555) 123-4567')

# 2. Middlewere